In [1]:
from datetime import datetime
from pathlib import Path
import tarfile

import torch
import pandas as pd

import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger

from src.dataset import ASRDataset
from src.normalizers.digit_char_normalizer import DigitCharNormalizer
from src.normalizers.unspittable_normalizer import UnspittableNormalizer
from src.normalizers.word_normalizer import RussianWordNormalizer
from src.augmenter import Augmenter

# Configuration

In [10]:
SAMPLE_RATE = 16000
N_MELS = 80
N_FFT = 256
HOP_LENGTH = 160

BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

dataset_path = base_dir = Path("data")
train_dataset_dir = (base_dir / "train") 
dev_dataset_dir =(base_dir / "dev")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading data

In [3]:
train_dataset_dir.mkdir(parents=True, exist_ok=True)
dev_dataset_dir.mkdir(parents=True, exist_ok=True)

train_csv = base_dir / "train.csv"
dev_csv = base_dir / "dev.csv"

# Extract datasets if not already extracted
if not train_csv.is_file():
    with tarfile.open(train_dataset_dir / "train.tar.gz", "r:gz") as tar:
        tar.extractall(dataset_path)

if not dev_csv.is_file():
    with tarfile.open(dev_dataset_dir / "dev.tar.gz", "r:gz") as tar:
        tar.extractall(dataset_path)

In [4]:
normalizer = UnspittableNormalizer()
vocab, vocab_size = normalizer.get_vocab()

print(vocab)

[0, 1000, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900]


In [5]:
augmenter = Augmenter(
    sample_rate=SAMPLE_RATE,
    noise_prob=0.5,
    speed_prob=0.3,
    gain_prob=0.4,
    shift_prob=0.5,
    clip_prob=0.4,
    reverb_prob=0.3
)

train_dataset = ASRDataset(
    csv_path=train_csv,
    root_dir=dataset_path,
    augmenter=augmenter,
    normalizer=normalizer,
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    device=device,
)

dev_dataset = ASRDataset(
    csv_path=dev_csv,
    root_dir=dataset_path,
    augmenter=None,
    normalizer=normalizer,
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    device=device,
)

e:\AnacondaNew\envs\ml-course-spbu\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(
Generating label files: 100%|██████████| 12553/12553 [00:00<00:00, 16555.76it/s]
e:\AnacondaNew\envs\ml-course-spbu\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(
Generating label files: 100%|██████████| 2265/2265 [00:00<00:00, 26362.07it/s]


## Final DataLoaders

In [6]:
def collate_fn(batch):
    mels, labels = zip(*batch)
    mel_lenghts = torch.tensor([mel.shape[0] for mel in mels])
    mels = torch.nn.utils.rnn.pad_sequence(mels, batch_first=True)

    labels_lengths = torch.tensor([len(label) for label in labels])
    # labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True)
    labels = torch.cat(labels)
    return mels, labels, mel_lenghts, labels_lengths

In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

## Check dataset view (Optional)

In [8]:
# test_dataset = ASRDataset(
#     csv_path=base_dir / "test.csv",
#     root_dir=dataset_path,
#     augmenter=Augmenter(),
#     normalizer=normalizer,
#     sample_rate=SAMPLE_RATE,
#     n_mels=N_MELS,
#     n_fft=N_FFT,
#     hop_length=HOP_LENGTH,
#     device=device,
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=collate_fn,
#     num_workers=0
# )

# for x, texts, lengths, t_lengths in test_loader:
#     print(f"{x.shape=}, {texts=}, {lengths=}")

# Model training

In [11]:
from src.models.simple_cnn import ASRModel
from src.trainer import LightningASR

model = ASRModel(vocab_size)
lit_model = LightningASR(model, vocab, lr=LR)

logger = CSVLogger(
    "logs", name=f"model_{datetime.utcnow().strftime('%Y-%m-%d_%H-%M-%S')}"
)
logger.log_hyperparams(
    {
        "sample_rate": SAMPLE_RATE,
        "n_mels": N_MELS,
        "n_fft": N_FFT,
        "hop_lenght": HOP_LENGTH,
        "batch": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "normalizer": str(normalizer),
        "augmentation": augmenter.get_params(),
    }
)

In [12]:
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices=1,
    log_every_n_steps=100,
    logger=logger,
)

trainer.fit(lit_model, train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type     | Params
--------------------------------------
0 | model    | ASRModel | 4.8 M 
1 | ctc_loss | CTCLoss  | 0     
--------------------------------------
4.8 M     Trainable params
0         Non-trainable params
4.8 M     Total params
19.060    Total estimated model params size (MB)


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


# Evaluate results

In [13]:
train_df = pd.read_csv(train_csv)
t_spk_ids = train_df["spk_id"].tolist()

dev_df = pd.read_csv(dev_csv)
d_spk_ids = dev_df["spk_id"].tolist()

in_domain_speakers = set(t_spk_ids) & set(d_spk_ids)
out_of_domain_speakers = set(d_spk_ids) - set(t_spk_ids)

In [14]:
from collections import defaultdict
import editdistance


def ctc_decode(sequence):
    result = []
    prev = None

    for t in sequence:
        if t != prev and t != 0:  # remove repeats + blanks
            result.append(t)
        prev = t

    return result

def cer(pred, target):
    return editdistance.eval(pred, target) / max(1, len(target))

def evaluate_by_speaker(
    model, dataloader, normalizer, spk_ids, in_domain_speakers, device=None
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    # metrics
    spk_cer = defaultdict(list)
    in_domain_cer = []
    out_domain_cer = []

    sample_idx = 0

    with torch.no_grad():
        for x, texts, lengths, t_lengths in dataloader:
            x = x.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=-1)

            offset = 0

            for i in range(len(t_lengths)):
                pred_tokens = preds[i].cpu().numpy()
                pred_tokens = ctc_decode(pred_tokens)
                pred_text = normalizer.tokens2num(pred_tokens)

                length = t_lengths[i]
                gt_tokens = texts[offset : offset + length].cpu().numpy()
                offset += length
                gt_text = normalizer.tokens2num(gt_tokens)

                # metrics
                spk = spk_ids[sample_idx]
                sample_idx += 1

                sample_cer = cer(pred_text, gt_text)

                spk_cer[spk].append(sample_cer)

                if spk in in_domain_speakers:
                    in_domain_cer.append(sample_cer)
                else:
                    out_domain_cer.append(sample_cer)

    # aggregate metrics
    spk_cer_mean = {spk: sum(vals) / len(vals) for spk, vals in spk_cer.items()}
    in_domain_mean = sum(in_domain_cer) / len(in_domain_cer) if in_domain_cer else None
    out_domain_mean = (
        sum(out_domain_cer) / len(out_domain_cer) if out_domain_cer else None
    )

    return {
        "per_speaker": spk_cer_mean,
        "in_domain": in_domain_mean,
        "out_of_domain": out_domain_mean,
    }

In [ ]:
results = evaluate_by_speaker(
    lit_model.model,
    dev_loader,
    normalizer,
    d_spk_ids,
    in_domain_speakers
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

score = ((2 * results["in_domain"] * results["out_of_domain"]) / (results["in_domain"] + results["out_of_domain"])) * 100
print(f"Harmonic CER mean: {score}%")

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")


In-domain CER: 0.16694444444444456
Out-of-domain CER: 0.1781581581581578
Harmonic CER mean: 17.236911291108793%

Per speaker:
spk_J 0.1629
spk_I 0.1885
spk_K 0.2370
spk_H 0.1605
spk_F 0.1650
spk_C 0.1870
spk_D 0.1763
spk_B 0.1202
spk_E 0.1357
spk_A 0.2175


In [14]:
results = evaluate_by_speaker(
    lit_model.model,
    train_loader,
    normalizer,
    t_spk_ids,
    t_spk_ids
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")

In-domain CER: 0.008942085557237352
Out-of-domain CER: None

Per speaker:
spk_E 0.0088
spk_B 0.0090
spk_A 0.0095
spk_C 0.0103
spk_D 0.0065
spk_F 0.0102
